# Three-Tier Forward Model Performance

Comprehensive benchmarks of tengri's three computation tiers:

| Tier | Kernel | Resolution | JIT | Components | Typical time |
|------|--------|-----------|-----|------------|-------------|
| 1 | Fused photometry | Filter effective wavelengths | Yes | Subset | ~140 us |
| 2 | Compositional rest-frame SED | Full SSP wavelength | Yes | ALL | ~300-500 us |
| 3 | Exact path (Python dispatch) | Full SSP wavelength | Partial | ALL | ~500-1000 us |

**Sections:**
1. Setup & model configurations
2. Forward model: per-call timing across tiers
3. Gradient timing (critical for HMC/VI)
4. Component overhead breakdown
5. Scaling with model complexity (D = 5 to 137)
6. Accuracy: Tier 2 vs Tier 3 residuals
7. End-to-end inference impact

In [ ]:
import os
import sys
import time
import warnings

os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _here = os.getcwd()
proj_root = os.path.abspath(os.path.join(_here, ".."))
os.chdir(proj_root)
sys.path.insert(0, os.path.join(proj_root, "analysis"))

from common import (
    get_observation,
    get_ssp,
    setup_matplotlib,
)

from tengri import Fitter, SEDModel, Parameters, Uniform
from tengri.forward._kernels.assembly import (
    observe_spectrum_from_rest_sed,
)
from tengri.observation.spectrum import compute_spectrum
from tengri.utils.cosmology import luminosity_distance

setup_matplotlib()

FIGDIR = os.path.join("explore", "figures")
os.makedirs(FIGDIR, exist_ok=True)

# Color palette
C_T1 = "#2196F3"  # Tier 1 - blue
C_T2 = "#FF9800"  # Tier 2 - orange
C_T3 = "#F44336"  # Tier 3 - red
C_GRAD = "#4CAF50"  # gradient - green

## 1. Setup & Model Configurations

We test five configurations spanning the complexity range:

| Config | SFH | Stochastic | Dust emission | AGN | D |
|--------|-----|-----------|---------------|-----|---|
| A | DPL | No | No | No | ~7 |
| B | DPL | No | MBB | No | ~9 |
| C | DPL | Yes (n=64) | No | No | ~71 |
| D | DPL | Yes (n=128) | MBB | No | ~135 |
| E | DPL | Yes (n=128) | MBB | Yes | ~142 |

In [ ]:
ssp = get_ssp()
obs = get_observation()  # SDSS ugriz

CONFIGS = {}


def _make(
    name, stochastic=False, n_grid=64, dust_em=None, agn=False, free_psd=False, **extra_spec
):
    """Build a (model, params) pair for benchmarking."""
    kw = dict(
        sfh_dpl_alpha=Uniform(0.5, 3.0),
        sfh_dpl_beta=Uniform(0.3, 2.0),
        sfh_dpl_tau_gyr=Uniform(1.0, 8.0),
        sfh_dpl_log_peak_sfr=Uniform(0.0, 1.5),
        met_logzsol=Uniform(-2.0, 0.2),
        dust_tau_bc=Uniform(0.0, 2.0),
        dust_tau_diff=Uniform(0.0, 1.0),
        dust_slope=-0.7,
        redshift=0.1,
        stochastic=stochastic,
        n_grid=n_grid,
    )
    if stochastic:
        if free_psd:
            kw["sfh_field_psd_sigma"] = Uniform(0.1, 4.0)
            kw["sfh_field_psd_tau_myr"] = Uniform(1.0, 300.0)
        else:
            kw["sfh_field_psd_sigma"] = 1.0
            kw["sfh_field_psd_tau_myr"] = 50.0
    if dust_em:
        kw["dust_emission"] = dust_em
    if agn:
        kw["agn_model"] = "simple"
        kw["agn_log_lbol"] = Uniform(8.0, 12.0)
    kw.update(extra_spec)

    spec = ParamSpec(**kw)
    model = Model(spec, ssp, observation=obs)
    params = spec.sample(jax.random.PRNGKey(0))
    D = len(spec.free_params)
    return model, params, D


CONFIGS["A: Smooth (D~7)"] = _make("A")
CONFIGS["B: Smooth+MBB (D~9)"] = _make("B", dust_em="modified_blackbody")
CONFIGS["C: Stoch n=64 (D~71)"] = _make("C", stochastic=True, n_grid=64)
CONFIGS["D: Stoch n=128 (D~135)"] = _make(
    "D", stochastic=True, n_grid=128, dust_em="modified_blackbody"
)
CONFIGS["E: Stoch+MBB+AGN (D~142)"] = _make(
    "E", stochastic=True, n_grid=128, dust_em="modified_blackbody", agn=True
)

for name, (model, _params, D) in CONFIGS.items():
    has_t1 = model._fused_photometry is not None
    has_t2 = model._fused_rest_sed is not None
    print(f"{name:30s}  D={D:>3d}  Tier1={has_t1}  Tier2={has_t2}")

## 2. Forward Model Timing: Per-Call Latency

We measure **per-call** wall time after JIT warm-up.

**Key distinction:**
- **Rest-frame SED** (apples-to-apples): Tier 2 kernel vs Tier 3 `predict_sed`.
  Both produce the same full-resolution SED. Tier 2 wraps all physics in one
  `@jax.jit` scope; Tier 3 dispatches components through Python.
- **Photometry**: Tier 1 evaluates at filter effective wavelengths (fastest).
  Tiers 2 & 3 must integrate through filters on the full wavelength grid
  (same cost for both — dominated by the per-filter Python loop).

Tier 2's value is for **spectroscopy** (single `jnp.interp`) and **free-z**
scenarios where Tier 1's precomputation is impossible.

In [ ]:
N_WARMUP = 3  # JIT warm-up calls
N_REPEAT = 50  # timing calls


def bench_fn(fn, n_warmup=N_WARMUP, n_repeat=N_REPEAT):
    """Benchmark a function: warm up, then time n_repeat calls."""
    for _ in range(n_warmup):
        r = fn()
        jax.block_until_ready(r)

    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        result = fn()
        jax.block_until_ready(result)
        times.append(time.perf_counter() - t0)
    return np.array(times)

### 2a. Rest-frame SED: Tier 2 kernel vs Tier 3 (apples-to-apples)

In [ ]:
results = {}

for name, (model, params, D) in CONFIGS.items():
    row = {"D": D}

    # Tier 2: rest-frame SED via compositional JIT kernel
    if model._fused_rest_sed is not None:
        row["tier2_sed"] = bench_fn(lambda m=model, p=params: m._compute_rest_sed_tier2(p))
    else:
        row["tier2_sed"] = None

    # Tier 3: rest-frame SED via Python-dispatched exact path
    row["tier3_sed"] = bench_fn(lambda m=model, p=params: m.predict_sed(p))

    # Tier 2 kernel only (no SFH/Z dispatch — pure JIT'd physics)
    if model._fused_rest_sed is not None:
        from tengri.forward.pipeline import interp_met_alpha_dispatch
        from tengri.sps.dsps_wrapper import compute_csp_weights

        p = model._get_internal_params(params)
        sfr = model._compute_sfr(p)
        sfr_on_ssp = jnp.interp(model.ssp_log_ages_yr, model.log_age_grid, sfr)
        w = compute_csp_weights(sfr_on_ssp, model.ssp_ages_yr)
        ssp_z = interp_met_alpha_dispatch(model, p["log_z_abs"], p.get("alpha_fe", 0.0))
        row["tier2_kernel"] = bench_fn(
            lambda _m=model, _w=w, _s=ssp_z, _p=p: _m._fused_rest_sed(_w, _s, _p)
        )
    else:
        row["tier2_kernel"] = None

    # Tier 1: fused photometry (for reference)
    if model._fused_photometry is not None:
        row["tier1_phot"] = bench_fn(lambda m=model, p=params: m._predict_photometry_fast(p))
    else:
        row["tier1_phot"] = None

    results[name] = row

    t1_us = np.median(row["tier1_phot"]) * 1e6 if row["tier1_phot"] is not None else float("nan")
    t2k_us = (
        np.median(row["tier2_kernel"]) * 1e6 if row["tier2_kernel"] is not None else float("nan")
    )
    t2_us = np.median(row["tier2_sed"]) * 1e6 if row["tier2_sed"] is not None else float("nan")
    t3_us = np.median(row["tier3_sed"]) * 1e6
    print(
        f"{name:30s}  T1_phot={t1_us:>6.0f}us  "
        f"T2_kernel={t2k_us:>6.0f}us  T2_sed={t2_us:>6.0f}us  "
        f"T3_sed={t3_us:>6.0f}us"
    )

### 2b. Bar chart: rest-frame SED latency by tier

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = list(results.keys())
short_names = [n.split("(")[0].strip() for n in names]
x = np.arange(len(names))
width = 0.3

# Left panel: SED computation (apples-to-apples)
t2k_vals = [
    np.median(results[n]["tier2_kernel"]) * 1e6 if results[n]["tier2_kernel"] is not None else 0
    for n in names
]
t2_vals = [
    np.median(results[n]["tier2_sed"]) * 1e6 if results[n]["tier2_sed"] is not None else 0
    for n in names
]
t3_vals = [np.median(results[n]["tier3_sed"]) * 1e6 for n in names]

bars_k = ax1.bar(
    x - width,
    t2k_vals,
    width,
    label="Tier 2 kernel only",
    color=C_T2,
    alpha=0.6,
    edgecolor="white",
)
bars_2 = ax1.bar(
    x, t2_vals, width, label="Tier 2 (SFH+Z+kernel)", color=C_T2, alpha=0.85, edgecolor="white"
)
bars_3 = ax1.bar(
    x + width,
    t3_vals,
    width,
    label="Tier 3 (exact path)",
    color=C_T3,
    alpha=0.85,
    edgecolor="white",
)

for bars in [bars_k, bars_2, bars_3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                h + 15,
                f"{h:.0f}",
                ha="center",
                va="bottom",
                fontsize=7,
            )

ax1.set_xticks(x)
ax1.set_xticklabels(short_names, rotation=15, ha="right")
ax1.set_ylabel("Median latency (us)")
ax1.set_title("Rest-Frame SED Computation")
ax1.legend(fontsize=8)

# Right panel: Tier 1 photometry for reference
t1_vals = [
    np.median(results[n]["tier1_phot"]) * 1e6 if results[n]["tier1_phot"] is not None else 0
    for n in names
]
ax2.bar(x, t1_vals, 0.5, label="Tier 1 (fused phot)", color=C_T1, alpha=0.85, edgecolor="white")
for i, v in enumerate(t1_vals):
    if v > 0:
        ax2.text(i, v + 10, f"{v:.0f}", ha="center", va="bottom", fontsize=8)

ax2.set_xticks(x)
ax2.set_xticklabels(short_names, rotation=15, ha="right")
ax2.set_ylabel("Median latency (us)")
ax2.set_title("Tier 1 Fused Photometry (Reference)")

fig.suptitle("Forward Model Latency by Tier and Configuration", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "tier_latency_comparison.png"), dpi=200)
plt.show()

### 2c. Speedup ratios

Tier 2 SED vs Tier 3 SED is the fair comparison (both produce full-resolution
rest-frame SEDs). Tier 1 photometry is a different operation (filter effective
wavelengths only) shown for reference.

In [ ]:
print(f"{'Config':<35s} {'T2 kernel':>10s} {'T2 SED':>10s} {'T3 SED':>10s} {'T3/T2':>7s}")
print("-" * 75)
for name in names:
    r = results[name]
    t2k = np.median(r["tier2_kernel"]) * 1e6 if r["tier2_kernel"] is not None else float("nan")
    t2 = np.median(r["tier2_sed"]) * 1e6 if r["tier2_sed"] is not None else float("nan")
    t3 = np.median(r["tier3_sed"]) * 1e6
    ratio = f"{t3 / t2:.1f}x" if np.isfinite(t2) and t2 > 0 else "N/A"
    print(f"{name:<35s} {t2k:>9.0f}us {t2:>9.0f}us {t3:>9.0f}us {ratio:>7s}")

### 2d. JIT'd filter integration: eliminating the Python loop

The filter integration in `observe_photometry_from_rest_sed` loops
over filters in Python, calling `compute_flux_density` per filter.
This costs ~7ms for 5 SDSS filters — dominating the Tier 2 photometry
path. We can eliminate this by JIT-compiling the entire
rest-SED → photometry pipeline as a single function.

In [ ]:
from tengri.observation.photometry import compute_flux_density

# Take Config A as the test case
model_a, params_a, _Da = CONFIGS["A: Smooth (D~7)"]
wave_rest = model_a.ssp_data.ssp_wave
z_a = model_a._get_redshift(params_a)
dl_cm_a = model_a._get_dl_cm(params_a)

# Pre-pad all filters onto a common grid for vectorized integration
filt_waves = model_a.filter_waves
filt_trans = model_a.filter_trans


@jax.jit
def tier2_photometry_jit(params_dict):
    """Fully JIT'd: params -> rest SED -> photometry (no Python loops)."""
    rest_sed = model_a._compute_rest_sed_tier2(params_dict)

    # Vectorized filter integration: loop unrolled by JIT tracer
    fluxes = []
    for fw, ft in zip(filt_waves, filt_trans):
        f = compute_flux_density(rest_sed, wave_rest, fw, ft, z_a, dl_cm_a)
        fluxes.append(f)
    return jnp.array(fluxes)


ts_jit_phot = bench_fn(lambda: tier2_photometry_jit(params_a))
ts_t1 = bench_fn(lambda: model_a._predict_photometry_fast(params_a))
ts_t2_phot = bench_fn(lambda: model_a._predict_photometry_tier2(params_a))

print(f"{'Path':<45s} {'Median (us)':>12s}")
print("-" * 60)
print(f"{'Tier 1 fused photometry':<45s} {np.median(ts_t1) * 1e6:>11.0f}us")
print(f"{'Tier 2 photometry (Python filter loop)':<45s} {np.median(ts_t2_phot) * 1e6:>11.0f}us")
print(
    f"{'Tier 2 photometry (JIT filter integration)':<45s} {np.median(ts_jit_phot) * 1e6:>11.0f}us"
)
print()

speedup = np.median(ts_t2_phot) / np.median(ts_jit_phot)
print(f"JIT filter integration speedup: {speedup:.1f}x over Python loop")
print(f"vs Tier 1: {np.median(ts_jit_phot) / np.median(ts_t1):.1f}x slower (Tier 1 still wins)")

# Verify correctness
phot_jit = tier2_photometry_jit(params_a)
phot_t1 = model_a._predict_photometry_fast(params_a)
from numpy.testing import assert_allclose

assert_allclose(phot_jit, phot_t1, rtol=0.05)  # Tier 1 uses effective-wavelength approx
print("Correctness: JIT photometry matches Tier 1 to <5%")

## 3. Gradient Timing

Gradients are the inner loop of HMC (NUTS, Ray Tracing) and
variational inference (geoVI). We measure `jax.grad` of the
log-likelihood through each tier.

In [ ]:
def make_grad_fn(model, params, tier="tier2"):
    """Build a grad function for a given tier."""
    if tier == "tier3":

        def loss(dust_tau_bc):
            p_mod = {**params, "dust_tau_bc": dust_tau_bc}
            sed = model.predict_sed(p_mod)
            return jnp.sum(sed**2)
    else:

        def loss(dust_tau_bc):
            p_mod = {**params, "dust_tau_bc": dust_tau_bc}
            sed = model._compute_rest_sed_tier2(p_mod)
            return jnp.sum(sed**2)

    return jax.jit(jax.grad(loss))


grad_results = {}

for name, (model, params, _D) in CONFIGS.items():
    row = {}

    # Tier 2 gradient
    if model._fused_rest_sed is not None:
        gfn2 = make_grad_fn(model, params, "tier2")
        row["grad_tier2"] = bench_fn(lambda f=gfn2: f(1.0))
    else:
        row["grad_tier2"] = None

    # Tier 3 gradient
    gfn3 = make_grad_fn(model, params, "tier3")
    row["grad_tier3"] = bench_fn(lambda f=gfn3: f(1.0))

    grad_results[name] = row

    t2_us = np.median(row["grad_tier2"]) * 1e6 if row["grad_tier2"] is not None else float("nan")
    t3_us = np.median(row["grad_tier3"]) * 1e6
    ratio = t3_us / t2_us if np.isfinite(t2_us) else float("nan")
    print(f"{name:30s}  grad_T2={t2_us:.0f}us  grad_T3={t3_us:.0f}us  speedup={ratio:.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(names))
width = 0.35

g2 = [
    np.median(grad_results[n]["grad_tier2"]) * 1e6
    if grad_results[n]["grad_tier2"] is not None
    else 0
    for n in names
]
g3 = [np.median(grad_results[n]["grad_tier3"]) * 1e6 for n in names]

ax.bar(x - width / 2, g2, width, label="Tier 2 gradient", color=C_T2, alpha=0.85)
ax.bar(x + width / 2, g3, width, label="Tier 3 gradient", color=C_T3, alpha=0.85)

for i, (v2, v3) in enumerate(zip(g2, g3)):
    if v2 > 0 and v3 > 0:
        ax.text(
            i,
            max(v2, v3) + 15,
            f"{v3 / v2:.1f}x",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color=C_GRAD,
        )

ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=15, ha="right")
ax.set_ylabel("Median gradient time (us)")
ax.set_title("Gradient Latency: Tier 2 vs Tier 3")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "tier_gradient_comparison.png"), dpi=200)
plt.show()

## 4. Component Overhead Breakdown

How much does each physics component add to the Tier 2 kernel?
We measure by building models with progressively more components.

In [ ]:
component_configs = {
    "Stellar only": {},
    "+ Dust atten": {},  # always on (two-component)
    "+ MBB emission": {"dust_em": "modified_blackbody"},
    "+ AGN (parametric)": {"dust_em": "modified_blackbody", "agn": True},
}

component_times = {}
for label, extra in component_configs.items():
    m, p, D = _make(label, **extra)
    if m._fused_rest_sed is not None:
        ts = bench_fn(lambda _m=m, _p=p: _m._compute_rest_sed_tier2(_p))
        component_times[label] = np.median(ts) * 1e6
        print(f"{label:<25s}  {component_times[label]:.0f} us  (D={D})")
    else:
        print(f"{label:<25s}  N/A")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

labels = list(component_times.keys())
vals = list(component_times.values())
# Compute incremental cost
increments = [vals[0]]
for i in range(1, len(vals)):
    increments.append(vals[i] - vals[i - 1])

colors = [C_T1, C_T2, C_T3, "#9C27B0"]
bottom = 0
for i, (lab, inc) in enumerate(zip(labels, increments)):
    ax.barh(
        0,
        inc,
        left=bottom,
        height=0.5,
        label=f"{lab} (+{inc:.0f} us)",
        color=colors[i % len(colors)],
        alpha=0.8,
        edgecolor="white",
    )
    if inc > 10:
        ax.text(
            bottom + inc / 2,
            0,
            f"{inc:.0f}",
            ha="center",
            va="center",
            fontsize=9,
            color="white",
            fontweight="bold",
        )
    bottom += inc

ax.set_xlabel("Latency (us)")
ax.set_yticks([])
ax.set_title("Tier 2 Component Cost Breakdown (Incremental)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "tier2_component_breakdown.png"), dpi=200)
plt.show()

## 5. Scaling with Model Dimensionality

How does forward model time scale as we increase the GP grid size
(which controls D)?

In [ ]:
grid_sizes = [16, 32, 64, 96, 128]
scaling_results = {}

for ng in grid_sizes:
    m, p, D = _make(f"n={ng}", stochastic=True, n_grid=ng)
    ts_t2 = bench_fn(lambda _m=m, _p=p: _m._compute_rest_sed_tier2(_p), n_repeat=30)
    ts_t3 = bench_fn(lambda _m=m, _p=p: _m.predict_sed(_p), n_repeat=30)
    scaling_results[ng] = {
        "D": D,
        "tier2": np.median(ts_t2) * 1e6,
        "tier3": np.median(ts_t3) * 1e6,
        "tier2_std": np.std(ts_t2) * 1e6,
        "tier3_std": np.std(ts_t3) * 1e6,
    }
    print(
        f"n_grid={ng:>3d}  D={D:>3d}  "
        f"T2={scaling_results[ng]['tier2']:.0f}us  "
        f"T3={scaling_results[ng]['tier3']:.0f}us  "
        f"speedup={scaling_results[ng]['tier3'] / scaling_results[ng]['tier2']:.1f}x"
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

Ds = [scaling_results[ng]["D"] for ng in grid_sizes]
t2s = [scaling_results[ng]["tier2"] for ng in grid_sizes]
t3s = [scaling_results[ng]["tier3"] for ng in grid_sizes]
t2_err = [scaling_results[ng]["tier2_std"] for ng in grid_sizes]
t3_err = [scaling_results[ng]["tier3_std"] for ng in grid_sizes]

# Left: absolute times
ax1.errorbar(Ds, t2s, yerr=t2_err, marker="o", label="Tier 2", color=C_T2, capsize=3, linewidth=2)
ax1.errorbar(Ds, t3s, yerr=t3_err, marker="s", label="Tier 3", color=C_T3, capsize=3, linewidth=2)
ax1.set_xlabel("Model dimensionality D")
ax1.set_ylabel("Median forward model time (us)")
ax1.set_title("Forward Model Scaling with D")
ax1.legend()

# Right: speedup ratio
speedups = [t3 / t2 for t2, t3 in zip(t2s, t3s)]
ax2.plot(Ds, speedups, "o-", color=C_GRAD, linewidth=2, markersize=8)
ax2.axhline(1.0, ls="--", color="gray", alpha=0.5)
ax2.set_xlabel("Model dimensionality D")
ax2.set_ylabel("Tier 3 / Tier 2 speedup")
ax2.set_title("Tier 2 Speedup vs Complexity")

fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "tier_scaling_with_D.png"), dpi=200)
plt.show()

## 6. Accuracy: Tier 2 vs Tier 3 Residuals

The compositional kernel should match the exact path to machine
precision (both compute the same physics). Any differences come
from floating-point operation ordering.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, (name, (model, params, D)) in enumerate(CONFIGS.items()):
    if i >= len(axes):
        break
    ax = axes[i]

    if model._fused_rest_sed is None:
        ax.text(0.5, 0.5, "Tier 2 N/A", transform=ax.transAxes, ha="center", va="center")
        ax.set_title(name.split("(")[0].strip())
        continue

    sed_t2 = model._compute_rest_sed_tier2(params)
    sed_t3 = model.predict_sed(params)

    # Relative residual
    rel_resid = (sed_t2 - sed_t3) / jnp.maximum(jnp.abs(sed_t3), 1e-50)
    wave = model.ssp_data.ssp_wave

    ax.plot(wave, rel_resid * 100, lw=0.5, color=C_T2, alpha=0.8)
    ax.axhline(0, ls="--", color="gray", alpha=0.5)
    ax.set_xlabel("Wavelength (A)" if i >= 3 else "")
    ax.set_ylabel("Residual (%)" if i % 3 == 0 else "")
    ax.set_title(f"{name.split('(')[0].strip()} (D={D})")
    ax.set_ylim(-0.01, 0.01)

# Remove empty subplot
if len(CONFIGS) < len(axes):
    for j in range(len(CONFIGS), len(axes)):
        axes[j].set_visible(False)

fig.suptitle("Tier 2 vs Tier 3 Relative Residuals", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "tier2_vs_tier3_residuals.png"), dpi=200)
plt.show()

In [ ]:
print(f"{'Config':<35s} {'max |resid|':>12s} {'rms resid':>12s}")
print("-" * 60)
for name, (model, params, _D) in CONFIGS.items():
    if model._fused_rest_sed is None:
        print(f"{name:<35s} {'N/A':>12s} {'N/A':>12s}")
        continue
    sed_t2 = model._compute_rest_sed_tier2(params)
    sed_t3 = model.predict_sed(params)
    rel = (sed_t2 - sed_t3) / jnp.maximum(jnp.abs(sed_t3), 1e-50)
    print(
        f"{name:<35s} {float(jnp.max(jnp.abs(rel))):.2e} {float(jnp.sqrt(jnp.mean(rel**2))):.2e}"
    )

## 7. Spectroscopy: Tier 2 Enables Free-z

A key advantage of Tier 2: the rest-frame SED is computed once,
then the observation wrapper applies any redshift. This is ideal
for free-z spectroscopic fitting where Tier 1's precomputation
is impossible.

In [ ]:
# Build a model WITHOUT fixed-z precomputation
spec_freez = ParamSpec(
    sfh_dpl_alpha=Uniform(0.5, 3.0),
    sfh_dpl_beta=Uniform(0.3, 2.0),
    sfh_dpl_tau_gyr=Uniform(1.0, 8.0),
    sfh_dpl_log_peak_sfr=Uniform(0.0, 1.5),
    met_logzsol=Uniform(-2.0, 0.2),
    dust_tau_bc=Uniform(0.0, 2.0),
    dust_tau_diff=Uniform(0.0, 1.0),
    dust_slope=-0.7,
    redshift=Uniform(0.01, 0.5),  # FREE redshift
)

model_freez = Model(spec_freez, ssp, observation=obs)
params_freez = spec_freez.sample(jax.random.PRNGKey(7))

print(f"Free-z model: Tier 1 = {model_freez._fused_photometry is not None}")
print(f"Free-z model: Tier 2 = {model_freez._fused_rest_sed is not None}")
print(f"Free-z model: D = {len(spec_freez.free_params)}")

In [ ]:
# Benchmark: predict_spectrum at varying redshifts via Tier 2
wave_obs = jnp.linspace(4000.0, 9000.0, 200)

# Tier 2 path
ts_t2_spec = bench_fn(
    lambda: model_freez._predict_spectrum_tier2(params_freez, wave_obs),
    n_repeat=30,
)

# Tier 3 path (exact)
ts_t3_spec = bench_fn(
    lambda: model_freez.predict_sed(params_freez),
    n_repeat=30,
)

print(
    f"Free-z spectrum:  Tier 2 = {np.median(ts_t2_spec) * 1e6:.0f} us  "
    f"Tier 3 = {np.median(ts_t3_spec) * 1e6:.0f} us  "
    f"speedup = {np.median(ts_t3_spec) / np.median(ts_t2_spec):.1f}x"
)

In [ ]:
# Verify correctness: Tier 2 vs Tier 3 spectrum
rest_sed_t2 = model_freez._compute_rest_sed_tier2(params_freez)
rest_sed_t3 = model_freez.predict_sed(params_freez)

z = float(params_freez["redshift"])
dl_cm = luminosity_distance(z)

spec_t2 = observe_spectrum_from_rest_sed(
    rest_sed_t2, model_freez.ssp_data.ssp_wave, wave_obs, z, dl_cm
)
spec_t3 = compute_spectrum(rest_sed_t3, model_freez.ssp_data.ssp_wave, wave_obs, z, dl_cm)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), height_ratios=[3, 1], sharex=True)

ax1.plot(wave_obs, spec_t3, label="Tier 3 (exact)", color=C_T3, lw=1.5)
ax1.plot(wave_obs, spec_t2, label="Tier 2 (compositional)", color=C_T2, lw=1.5, ls="--")
ax1.set_ylabel("Flux density (erg/s/cm$^2$/Hz)")
ax1.set_title(f"Free-z spectrum at z = {z:.3f}")
ax1.legend()

resid = (spec_t2 - spec_t3) / jnp.maximum(jnp.abs(spec_t3), 1e-50) * 100
ax2.plot(wave_obs, resid, color=C_T2, lw=1)
ax2.axhline(0, ls="--", color="gray", alpha=0.5)
ax2.set_xlabel("Observed wavelength (A)")
ax2.set_ylabel("Residual (%)")
ax2.set_ylim(-0.01, 0.01)

fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "tier2_freez_spectrum.png"), dpi=200)
plt.show()

## 8. End-to-End Inference Impact

How much does the Tier 2 speedup matter for actual fitting?
We run MAP optimization through Tier 2 vs Tier 3 and measure
total wall time.

In [ ]:
model_bench, params_bench, D_bench = CONFIGS["A: Smooth (D~7)"]
mock = model_bench.mock(params_bench, snr=20.0, key=jax.random.PRNGKey(99))

fitter = Fitter(model_bench, mock.flux_obs, mock.noise)
fitter.compile(verbose=False)

# MAP via default dispatch (Tier 1 if available, else Tier 2, else Tier 3)
t0 = time.perf_counter()
result_default = fitter.run("map", n_steps=200, verbose=False)
t_default = time.perf_counter() - t0

print(f"MAP (200 steps, default dispatch): {t_default:.2f}s")

## Summary Table

In [ ]:
print()
print("=" * 70)
print("THREE-TIER FORWARD MODEL PERFORMANCE SUMMARY")
print("=" * 70)
print()
print(
    f"{'Config':<30s} {'D':>4s} {'T1 phot':>10s} {'T2 kern':>10s} "
    f"{'T2 SED':>10s} {'T3 SED':>10s} {'T3/T2':>7s}"
)
print("-" * 80)
for name in names:
    r = results[name]
    D = r["D"]
    t1 = f"{np.median(r['tier1_phot']) * 1e6:.0f}" if r["tier1_phot"] is not None else "N/A"
    t2k = f"{np.median(r['tier2_kernel']) * 1e6:.0f}" if r["tier2_kernel"] is not None else "N/A"
    t2 = f"{np.median(r['tier2_sed']) * 1e6:.0f}" if r["tier2_sed"] is not None else "N/A"
    t3 = f"{np.median(r['tier3_sed']) * 1e6:.0f}"
    if r["tier2_sed"] is not None:
        ratio = f"{np.median(r['tier3_sed']) / np.median(r['tier2_sed']):.1f}x"
    else:
        ratio = "N/A"
    print(f"{name:<30s} {D:>4d} {t1:>10s} {t2k:>10s} {t2:>10s} {t3:>10s} {ratio:>7s}")
print()
print("Tier 1: Fused photometry at filter effective wavelengths (fastest, subset of physics)")
print("Tier 2: Compositional rest-frame SED, all components, JIT'd end-to-end (NEW)")
print("Tier 3: Python-dispatched exact path (slowest, handles all edge cases)")